In [15]:
import pandas as pd
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer

df = pd.read_csv('data/csv/movie_reviews.csv')

okt = Okt()

def preprocess(text):
    morphs = okt.pos(text, stem=True)
    clean_words = [word for word, pos in morphs if pos in ['Noun','Adjective','Verb']]
    return ' '.join(clean_words)

df['clean_review'] = df['review'].astype(str).apply(preprocess)

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df['clean_review'])

print("벡터화된 데이터 크기:", tfidf_matrix.shape)

벡터화된 데이터 크기: (100, 381)


In [ ]:
import pandas as pd
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

df = pd.read_csv('data/csv/movie_reviews.csv')

okt = Okt()

def preprocess(text):
    morphs = okt.pos(text, stem=True)
    clean_words = [word for word, pos in morphs if pos in ['Noun','Adjective','Verb']]
    return ' '.join(clean_words)

df['clean_review'] = df['review'].astype(str).apply(preprocess)

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['clean_review'])
y = df['label'] 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression()
model.fit(X_train, y_train)

score = model.score(X_test, y_test)
print("테스트 정확도:", score)

text = '날씨가 화창한데 자격증 시험에 떨어졌습니다'
vector_text = vectorizer.transform([text])
res = model.predict(vector_text)
prob = model.predict_proba(vector_text)
negative_prob = prob[0][0]
positive_prob = prob[0][1]

print(f'{text} : {'긍정' if res[0] else '부정'}')
print(f"예측 상세 확률: 긍정({positive_prob:.2%}) / 부정({negative_prob:.2%})")

테스트 정확도: 0.65
날씨가 화창한데 자격증 시험에 떨어졌습니다 : 부정
예측 상세 확률: 긍정(46.95%) / 부정(53.05%)


In [ ]:
import pandas as pd
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.svm import SVC

df = pd.read_csv('data/csv/movie_reviews.csv')
df = df[df['review'].str.len() > 5]

okt = Okt()
def preprocess(text):
    morphs = okt.pos(str(text), stem=True)
    return ' '.join([word for word, pos in morphs if pos in ['Noun','Adjective','Verb']])

df['clean_review'] = df['review'].apply(preprocess)

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['clean_review'])
y = df['label']

kf = KFold(n_splits=5, shuffle=True, random_state=42)

param_grid = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf']
}
svc = SVC(probability=True)
grid = GridSearchCV(svc, param_grid, cv=kf, scoring='accuracy', n_jobs=-1)
grid.fit(X, y)

print("최적 하이퍼파라미터:", grid.best_params_)
print("최고 교차검증 정확도:", grid.best_score_)

test_text = '영화가 정말 재미있었어요!'
vector_text = vectorizer.transform([preprocess(test_text)])
res = grid.predict(vector_text)
proba = grid.predict_proba(vector_text)[0]

print(f'{test_text} : {"긍정" if res[0] else "부정"}')
print(f'예측 확률 - 부정: {proba[0]:.2f}%, 긍정: {proba[1]:.2f}%')

최적 하이퍼파라미터: {'C': 1, 'kernel': 'rbf'}
최고 교차검증 정확도: 0.71
영화가 정말 재미있었어요! : 긍정
예측 확률 - 부정: 0.10%, 긍정: 0.90%
